# Eval-determinism gate v2 — robosuite / LIBERO, in an isolated Python 3.11 venv

## Why v1 failed, exactly

Run 2026-09-12 in Colab. Two separate faults, both in cell 1:

1. **Colab is now Python 3.13** (`/usr/local/lib/python3.13`). `numpy<2` has no cp313 wheel, so
   the resolver silently overrode the `numpy==1.26.4` pin and installed **numpy 2.1.3**. `gym
   0.25.2` then warned it does not support NumPy 2. Resolved stack was numpy 2.1.3 / robosuite
   1.4.0 / **mujoco 3.13.0** / gym 0.25.2 — three of those are outside what LIBERO expects.
2. **`pip install git+… --no-deps` produced no importable package.** Cell 2 raised
   `ModuleNotFoundError: No module named 'libero'` and cells 3–8 aborted. The tests never ran.

**Neither was a result. v1 produced no evidence about LIBERO at all.**

## What changed in v2

- **The Python version is no longer Colab's.** `uv` builds a **3.11** venv, where every pin LIBERO
  asks for actually has a wheel. Nothing is installed into Colab's own interpreter.
- **LIBERO is `git clone`d and installed editable** (`uv pip install -e … --no-deps`), which
  produces a real importable package instead of relying on a wheel build from a git URL.
- **`mujoco<3.2`**, because robosuite 1.4.0 predates mujoco 3.x and v1 resolved to 3.13.0.
- **All five tests live in one script** `/content/gate.py`, run by `/content/venv/bin/python`. A
  notebook cell cannot import from another interpreter's venv; a subprocess can. This is the
  structural fix for the v1 failure, not a style change.
- **Resolved versions are printed from inside the venv before the tests run**, so a silently
  overridden pin is visible immediately rather than after eight cells.
- **EGL with an OSMesa fallback**: the runner tries `MUJOCO_GL=egl`, and on a non-zero exit
  re-runs once with `osmesa`.

## Pre-registered predictions — unchanged from v1, do not edit after running

| Test | Question | **Prediction** |
|---|---|---|
| **T1** | `seed(s)` then `reset()` twice, fresh envs → identical sim state? | **PASS** |
| **T2** | reset after K steps, no reseed → does the next episode depend on K? | **PASS** |
| **T3** | same, but `seed()` called again before each reset | **PASS** |
| **T4** | 6 consecutive resets vs 5 — is the 5-run a prefix of the 6-run? | **PASS** |
| **T5** | **burn 1000 `np.random.random()` between resets** → does the next episode change? | 🔴 **FAIL** |

**Why T2 is now predicted PASS:** the gymnasium run on 2026-09-12 19:32 UTC returned T2 PASS, and
the original step-count hypothesis was wrong — a PRNG that is *not reset* continues
deterministically. That prediction is corrected here rather than quietly dropped.

**T5 is the live hypothesis.** robosuite 1.4's placement samplers draw from the **global**
`np.random`, which any code in the process can advance, unlike a per-env PRNG. And
`ControlEnv.reset()` retries on `RandomizationError` in a `while` loop, so the randomness consumed
per reset is *variable*.

**Verdict rule, fixed in advance:** `CONFOUND CONFIRMED` iff T1 passes and (T2 fails **or** T5
fails) · `HARNESS SOUND` iff T1, T2 and T5 all pass — **in which case P6 is dead and gets recorded
as dead** · `INCONCLUSIVE` if any control disagrees with itself · `BROKEN` if T1 fails.


## Cell 1 — uv, a Python 3.11 venv, and the pinned stack

Expect several minutes. Read the `RESOLVED` block in cell 2, not the warnings here.

In [ ]:
import subprocess, sys, os

def sh(cmd, check=True):
    print(">>>", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = "\n".join(r.stdout.strip().splitlines()[-12:])
    print(tail, flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"FAILED ({r.returncode}): {cmd}")
    return r.returncode

sh(f"{sys.executable} -m pip install -q uv")
sh("uv venv /content/venv --python 3.11")

PY  = "/content/venv/bin/python"
UVP = "uv pip install --python /content/venv/bin/python"

sh(f"{UVP} 'numpy==1.26.4'")
sh(f"{UVP} 'robosuite==1.4.0' 'bddl==1.0.1' 'gym==0.25.2' easydict "
   f"opencv-python-headless 'mujoco<3.2'")

if not os.path.isdir("/content/LIBERO"):
    sh("git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO")
sh(f"{UVP} -e /content/LIBERO --no-deps")
print("\ninstall step finished")

## Cell 2 — resolved versions **from inside the venv**

This is the check v1 did not have. If `numpy` is not `1.26.x`, or `mujoco` is `3.2+`, or `libero`
fails to import, **stop here** — the tests below would produce a number about the wrong stack.


In [ ]:
probe = r"""
import sys, importlib
print("python      ", sys.version.split()[0])
for m in ("numpy","robosuite","mujoco","bddl","gym","libero"):
    try:
        mod = importlib.import_module(m)
        print(f"{m:12s}", getattr(mod, "__version__", "installed (no __version__)"))
    except Exception as e:
        print(f"{m:12s} IMPORT FAILED: {type(e).__name__}: {e}")
"""
open("/content/probe.py", "w").write(probe)
print(subprocess.run(["/content/venv/bin/python", "/content/probe.py"],
                     text=True, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT).stdout)

## Cell 3 — write the test script

In [ ]:
%%writefile /content/gate.py
"""T1-T5 eval-determinism gate on the real LIBERO / robosuite stack.

Runs inside the Python 3.11 venv. No policy, no checkpoint, no GPU: actions come from a
pinned RandomState and the fingerprint is LIBERO's own get_sim_state(), so any difference
in environment state is the environment.
"""
import os, sys, hashlib, json, datetime, traceback

BACKEND = os.environ.get("MUJOCO_GL", "egl")
import numpy as np

TASK_SUITE   = "libero_object"
TASK_NAME    = "pick_up_the_alphabet_soup_and_place_it_in_the_basket"
SEEDS        = [11, 12, 13]
K_SHORT, K_LONG = 40, 80
GLOBAL_DRAWS = 1000

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

_b     = benchmark.get_benchmark_dict()[TASK_SUITE]()
_names = [_b.get_task(i).name for i in range(_b.n_tasks)]
TASK_ID = _names.index(TASK_NAME)
_task   = _b.get_task(TASK_ID)
BDDL    = os.path.join(get_libero_path("bddl_files"), _task.problem_folder, _task.bddl_file)
print(f"[gate] backend={BACKEND} task={TASK_ID} {_task.name}", flush=True)

def mk():
    return OffScreenRenderEnv(bddl_file_name=BDDL, camera_heights=128, camera_widths=128)

def key(env):
    s = np.asarray(env.get_sim_state(), dtype=np.float64)
    return hashlib.sha256(np.ascontiguousarray(s).tobytes()).hexdigest()[:16]

def acts(k, tag, env):
    rs = np.random.RandomState(1234 if tag == "armA" else 5678)
    return [rs.uniform(-0.2, 0.2, size=env.env.action_dim) for _ in range(k)]

def seeded_once(s):
    e = mk(); e.seed(int(s)); e.reset(); k = key(e); e.close(); return k

def roll_reset(s, k, tag, reseed=None, burn=0):
    e = mk(); e.seed(int(s)); e.reset()
    for a in acts(k, tag, e):
        e.step(a)
    for _ in range(burn):
        np.random.random()                 # T5 only: advance the GLOBAL numpy RNG
    if reseed is not None:
        e.seed(int(reseed))
    e.reset()                              # unseeded unless reseed given
    kk = key(e); e.close(); return kk

def seq(s, n):
    e = mk(); e.seed(int(s)); e.reset(); out = [key(e)]
    for _ in range(n - 1):
        e.reset(); out.append(key(e))
    e.close(); return out

R = {"task": TASK_NAME, "seeds": SEEDS, "k_short": K_SHORT, "k_long": K_LONG,
     "global_draws": GLOBAL_DRAWS, "mujoco_gl": BACKEND,
     "predictions": {"T1": "PASS", "T2": "PASS", "T3": "PASS", "T4": "PASS",
                     "T5": "FAIL (pre-registered)"}}

print("\n== T1  seeded reset reproducibility ==", flush=True)
R["T1"] = {}
for s in SEEDS:
    a, b = seeded_once(s), seeded_once(s)
    R["T1"][s] = {"a": a, "b": b, "match": a == b}
    print(f"  seed {s}: {a} vs {b} -> {'MATCH' if a==b else 'DIFFER'}", flush=True)
T1_PASS = all(v["match"] for v in R["T1"].values())

print("\n== T2  does the next episode depend on STEPS taken? ==", flush=True)
R["T2"] = {}
for s in SEEDS:
    sh = roll_reset(s, K_SHORT, "armA")
    lo = roll_reset(s, K_LONG,  "armB")
    ct = roll_reset(s, K_SHORT, "armA")
    R["T2"][s] = {"short": sh, "long": lo, "control": ct,
                  "arms_match": sh == lo, "control_match": sh == ct}
    print(f"  seed {s}: K{K_SHORT}={sh} K{K_LONG}={lo} arms="
          f"{'MATCH' if sh==lo else 'DIFFER'} ctrl={'ok' if sh==ct else 'UNSTABLE'}", flush=True)
CONTROL_OK = all(v["control_match"] for v in R["T2"].values())
T2_PASS    = all(v["arms_match"]    for v in R["T2"].values())

print("\n== T3  does calling seed() again before each reset fix it? ==", flush=True)
R["T3"] = {}
for s in SEEDS:
    a = roll_reset(s, K_SHORT, "armA", reseed=s + 1)
    b = roll_reset(s, K_LONG,  "armB", reseed=s + 1)
    R["T3"][s] = {"a": a, "b": b, "match": a == b}
    print(f"  seed {s}: {a} vs {b} -> {'MATCH' if a==b else 'DIFFER'}", flush=True)
T3_PASS = all(v["match"] for v in R["T3"].values())

print("\n== T4  6-reset vs 5-reset sequence (the mhh-gate shape) ==", flush=True)
R["T4"] = {}
for s in SEEDS:
    six, five = seq(s, 6), seq(s, 5)
    R["T4"][s] = {"six": six, "five": five, "prefix_aligned": six[:5] == five,
                  "all_distinct": len(set(six)) == len(six)}
    print(f"  seed {s}: prefix_aligned={six[:5]==five} "
          f"all_distinct={len(set(six))==len(six)}", flush=True)
T4_ALIGNED = all(v["prefix_aligned"] for v in R["T4"].values())

print("\n== T5  GLOBAL numpy RNG probe -- the test this notebook exists for ==", flush=True)
R["T5"] = {}
for s in SEEDS:
    base = roll_reset(s, K_SHORT, "armA", burn=0)
    burn = roll_reset(s, K_SHORT, "armA", burn=GLOBAL_DRAWS)
    ctrl = roll_reset(s, K_SHORT, "armA", burn=0)
    R["T5"][s] = {"no_burn": base, "burned": burn, "control": ctrl,
                  "match": base == burn, "control_match": base == ctrl}
    print(f"  seed {s}: no_burn={base} burned={burn} -> "
          f"{'MATCH (global RNG unused)' if base==burn else 'DIFFER (GLOBAL RNG IS USED)'}"
          f" ctrl={'ok' if base==ctrl else 'UNSTABLE'}", flush=True)
T5_CONTROL_OK = all(v["control_match"] for v in R["T5"].values())
T5_PASS       = all(v["match"]         for v in R["T5"].values())

import importlib
ver = {}
for m in ("numpy", "robosuite", "mujoco", "bddl", "gym", "libero"):
    try:
        ver[m] = getattr(importlib.import_module(m), "__version__", "installed-no-__version__")
    except Exception as e:
        ver[m] = f"IMPORT FAILED: {type(e).__name__}"
ver["python"] = sys.version.split()[0]

if not (CONTROL_OK and T5_CONTROL_OK):
    verdict = "INCONCLUSIVE"
    reading = "A control disagreed with itself; the harness is noisy for a reason this test does not isolate."
elif not T1_PASS:
    verdict = "BROKEN"
    reading = "Seeded resets are not reproducible at all; suspect the install before LIBERO."
elif T2_PASS and T5_PASS:
    verdict = "HARNESS SOUND"
    reading = ("Neither step count nor global-RNG consumption changes the episode sequence. "
               "P6 is dead: record it as dead and do not revive it without a new mechanism.")
else:
    which = []
    if not T2_PASS: which.append("step count")
    if not T5_PASS: which.append("global-RNG consumption")
    verdict = "CONFOUND CONFIRMED"
    reading = ("The episode sequence depends on " + " and ".join(which) +
               ", so two arms differing in that respect are scored on different episodes. "
               "This is the paper.")

R.update({"timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
          "versions": ver, "verdict": verdict, "reading": reading,
          "T1_pass": T1_PASS, "T2_pass": T2_PASS, "T3_pass": T3_PASS,
          "T4_prefix_aligned": T4_ALIGNED, "T5_pass": T5_PASS,
          "control_ok": CONTROL_OK, "t5_control_ok": T5_CONTROL_OK})

with open("/content/det_result_robosuite.json", "w") as f:
    json.dump(R, f, indent=2, default=str)

print("\n" + "=" * 68)
print(json.dumps({k: R[k] for k in
                  ["timestamp_utc", "verdict", "reading", "versions", "T1_pass", "T2_pass",
                   "T3_pass", "T4_prefix_aligned", "T5_pass", "control_ok", "t5_control_ok",
                   "predictions"]}, indent=2))
print("=" * 68)
print("full record: /content/det_result_robosuite.json")


## Cell 4 — run it, EGL first then OSMesa

The script is run as a subprocess under `/content/venv/bin/python` because a Colab cell cannot
import from another interpreter's venv. On a non-zero exit the runner retries once with
`MUJOCO_GL=osmesa`, installing the OSMesa libraries first.


In [ ]:
import subprocess, os

def run_gate(backend):
    env = dict(os.environ, MUJOCO_GL=backend, PYOPENGL_PLATFORM=backend)
    p = subprocess.run(["/content/venv/bin/python", "/content/gate.py"],
                       env=env, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT)
    print(p.stdout)
    return p.returncode

rc = run_gate("egl")
if rc != 0:
    print("\n" + "=" * 68)
    print("EGL run failed (exit %d). Installing OSMesa and retrying once." % rc)
    print("=" * 68 + "\n")
    subprocess.run("apt-get -qq install -y libosmesa6-dev > /dev/null 2>&1", shell=True)
    rc = run_gate("osmesa")

print("\nexit code:", rc)
if rc == 0:
    print("PASTE THE JSON BLOCK ABOVE BACK for Paper Choice 2026-09-12.md section 15")
else:
    print("Both backends failed. Paste the traceback above; do not treat this as a result.")